# Lesson 4: Trees, Knowledge Graphs, and Network Graphs
In this lesson we explore graph representations of data: **trees**, **knowledge graphs**, and **network graphs**. We'll use **NetworkX** for analysis and **PyVis** for interactive visualizations.

Contents:
- Building graphs from edge lists and adjacency matrices
- Visualizing trees and directed graphs
- Computing network metrics (degree, centrality, clustering)
- Community detection and coloring
- Interactive visualization with PyVis
- A Mapper-style graph (simple implementation) and optional KeplerMapper example
- Exercises / Project ideas

> Recommended: run this notebook in Google Colab for best interactive support (PyVis HTML display).

## Setup: Install / Import Libraries

In [ ]:
# Install optional libraries (PyVis for interactive graphs, kmapper optional)
# Run these in Colab; on local machines you may skip installs if already available.
try:
    import pyvis
    print("pyvis already installed")
except Exception:
    !pip install pyvis

# kmapper is optional and can be heavy; we'll attempt to install but notebook also includes a simpler Mapper-like approach
try:
    import kmapper
    print("kmapper already installed")
except Exception:
    !pip install kmapper --quiet || true

# Common imports
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pyvis.network import Network

# display settings
plt.rcParams.update({'figure.max_open_warning': 0})
print('Imports ready')

## Building graphs: Edge lists and Adjacency matrices
A graph can be created from an **edge list** (rows of source-target pairs) or an **adjacency matrix** (square matrix).

In [ ]:
# Example edge list as a pandas DataFrame
edges_df = pd.DataFrame({
    'source': ['Alice','Alice','Bob','Charlie','David','Eve','Frank','Gina','Henry'],
    'target': ['Bob','Charlie','David','Eve','Eve','Frank','Gina','Henry','Alice'],
    'weight': [1,2,1,3,1,2,1,1,1]
})
edges_df.head()

In [ ]:
# Create graph from edge list
G = nx.from_pandas_edgelist(edges_df, 'source', 'target', ['weight'])
print("Nodes:", list(G.nodes()))
print("Edges:", list(G.edges(data=True)))

In [ ]:
# Example adjacency matrix
nodes = ['A','B','C','D']
adj = np.array([
    [0,1,0,1],
    [1,0,1,0],
    [0,1,0,1],
    [1,0,1,0]
])
adj_df = pd.DataFrame(adj, index=nodes, columns=nodes)
adj_df

In [ ]:
# Create graph from adjacency matrix
G2 = nx.from_numpy_array(adj)
# rename nodes to letters
mapping = {i:nodes[i] for i in range(len(nodes))}
G2 = nx.relabel_nodes(G2, mapping)
print("G2 nodes:", list(G2.nodes()))
print("G2 edges:", list(G2.edges()))

## Trees and Directed Graphs
Let's create a simple company hierarchy (directed tree) and visualize it.

In [ ]:
# Build a company hierarchy as a directed graph
tree_edges = [
    ('CEO','CTO'), ('CEO','CFO'), ('CEO','COO'),
    ('CTO','Dev Manager'), ('CTO','QA Manager'),
    ('Dev Manager','Dev1'), ('Dev Manager','Dev2'),
    ('QA Manager','QA1'), ('QA Manager','QA2'),
    ('CFO','Accountant')
]
T = nx.DiGraph()
T.add_edges_from(tree_edges)

# Plot using spring layout (not strictly hierarchical, but clear)
plt.figure(figsize=(10,6))
pos = nx.spring_layout(T, seed=42)
nx.draw(T, pos, with_labels=True, arrows=True, node_size=1800, node_color='lightblue', font_size=10)
plt.title('Company Hierarchy (Directed Tree)')
plt.show()

## Knowledge Graph (toy example)
Create a small knowledge graph connecting entities and types.

In [ ]:
kg_edges = [
    ('Paris','is_capital_of','France'),
    ('Berlin','is_capital_of','Germany'),
    ('Madrid','is_capital_of','Spain'),
    ('France','part_of','Europe'),
    ('Germany','part_of','Europe'),
    ('Spain','part_of','Europe'),
    ('Europe','contains','France'),
    ('Europe','contains','Germany'),
    ('Europe','contains','Spain')
]

# Create a simple multi-edge style graph: using node labels and a separate DataFrame for edges
kg_df = pd.DataFrame(kg_edges, columns=['source','relation','target'])
kg_df.head()

In [ ]:
# For visualization we simplify to undirected edges between entities (ignoring relation types for layout)
KG = nx.from_pandas_edgelist(kg_df[['source','target']], 'source','target')
plt.figure(figsize=(8,5))
pos = nx.spring_layout(KG, seed=7)
nx.draw(KG, pos, with_labels=True, node_color='lightgreen', node_size=1200)
# draw edge labels (relations)
edge_labels = {(row['source'], row['target']): row['relation'] for _,row in kg_df.iterrows()}
nx.draw_networkx_edge_labels(KG, pos, edge_labels=edge_labels, font_color='gray')
plt.title('Toy Knowledge Graph (relations displayed on edges)')
plt.show()

## Network Metrics: Degree, Centrality, Clustering Coefficient
Compute common graph metrics and visualize nodes sized/colored by metric values.

In [ ]:
# Use the friendship graph G from earlier
G_metrics = G.copy()

# Degree
deg = dict(G_metrics.degree())
nx.set_node_attributes(G_metrics, deg, 'degree')

# Betweenness centrality
bet = nx.betweenness_centrality(G_metrics)
nx.set_node_attributes(G_metrics, bet, 'betweenness')

# Clustering coefficient
clust = nx.clustering(G_metrics)
nx.set_node_attributes(G_metrics, clust, 'clustering')

# Build a visualization: node size by degree, color by betweenness
sizes = [200 + 400*deg[n] for n in G_metrics.nodes()]
colors = [bet[n] for n in G_metrics.nodes()]

plt.figure(figsize=(8,6))
pos = nx.spring_layout(G_metrics, seed=11)
nodes = nx.draw_networkx_nodes(G_metrics, pos, node_size=sizes, node_color=colors, cmap=plt.cm.viridis)
nx.draw_networkx_labels(G_metrics, pos)
nx.draw_networkx_edges(G_metrics, pos, alpha=0.4)
plt.colorbar(nodes, label='Betweenness centrality')
plt.title('Graph: node size ~ degree, color ~ betweenness')
plt.show()

## Community Detection and Coloring
We'll use the greedy modularity communities algorithm (NetworkX).

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(G))
# map node -> community index
comm_map = {}
for i, com in enumerate(communities):
    for node in com:
        comm_map[node] = i

# Prepare color list
colors = [comm_map[node] for node in G.nodes()]

plt.figure(figsize=(8,6))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color=colors, cmap=plt.cm.tab10, node_size=800)
plt.title(f'Communities detected (k={len(communities)}) via greedy_modularity_communities')
plt.show()

## Interactive Visualization with PyVis
PyVis creates interactive HTML visualizations you can open inside Colab or a browser.

In [ ]:
# Build a pyvis network from G
net = Network(height="600px", width="100%", notebook=True)
net.from_nx(G)
# optionally set physics layout
net.toggle_physics(True)
net.show("friendship_graph.html")
print("PyVis HTML generated: friendship_graph.html (click to open in Colab output)")

## Karate Club Graph: Example from Network Science
We use the classic Zachary's Karate Club graph to demonstrate metrics and communities.

In [ ]:
K = nx.karate_club_graph()
print(nx.info(K))

# compute degree and centrality
degK = dict(K.degree())
betK = nx.betweenness_centrality(K)
# community detection
commsK = list(greedy_modularity_communities(K))
print(f"Detected {len(commsK)} communities")

# draw with community colors
comm_map_K = {}
for i,com in enumerate(commsK):
    for n in com:
        comm_map_K[n] = i
colorsK = [comm_map_K[n] for n in K.nodes()]

plt.figure(figsize=(8,6))
pos = nx.spring_layout(K, seed=10)
nx.draw(K, pos, with_labels=True, node_color=colorsK, cmap=plt.cm.tab10, node_size=[100+20*degK[n] for n in K.nodes()])
plt.title('Zachary Karate Club: communities and degree-scaled nodes')
plt.show()

## Mapper-style Graph (simple implementation)
Mapper (from Topological Data Analysis) clusters points in overlapping intervals of a lens and connects clusters sharing points.
This is a lightweight, educational implementation that mimics the idea without requiring the full kmapper library.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

# Generate toy 2D data (moons)
X, y = make_moons(n_samples=300, noise=0.08, random_state=0)
scaler = MinMaxScaler()
Xs = scaler.fit_transform(X)

# Lens: use first coordinate (could be any projection)
lens = Xs[:,0]

# Define overlapping intervals on the lens
n_intervals = 8
overlap_perc = 0.4
interval_length = (lens.max() - lens.min()) / (n_intervals - (n_intervals-1)*overlap_perc)

intervals = []
start = lens.min()
for i in range(n_intervals):
    end = start + interval_length
    intervals.append((start, end))
    start = start + interval_length*(1-overlap_perc)

# Cluster points within each interval and create nodes
nodes = []
node_points = []
for i,(a,b) in enumerate(intervals):
    mask = (lens >= a) & (lens <= b)
    Xi = Xs[mask]
    if len(Xi) == 0:
        continue
    # cluster within interval
    k = max(2, min(6, len(Xi)//15))  # heuristic
    kmeans = KMeans(n_clusters=k, random_state=0).fit(Xi)
    for lbl in np.unique(kmeans.labels_):
        pts_idx = np.where(mask)[0][kmeans.labels_==lbl]
        nodes.append((i,lbl))
        node_points.append(set(pts_idx))

# Build graph: connect nodes with overlapping point sets
M = nx.Graph()
for idx, pts in enumerate(node_points):
    M.add_node(idx, size=len(pts))
for i in range(len(node_points)):
    for j in range(i+1, len(node_points)):
        if len(node_points[i].intersection(node_points[j]))>0:
            M.add_edge(i,j)

# Visualize Mapper-like graph
plt.figure(figsize=(8,6))
sizes = [50 + 8*M.nodes[n]['size'] for n in M.nodes()]
pos = nx.spring_layout(M, seed=5)
nx.draw(M, pos, with_labels=True, node_size=sizes, node_color='skyblue')
plt.title('Simple Mapper-like Graph (moons dataset)')
plt.show()

## Exercises & Project Ideas
1. Load a real-world edge list (e.g., social network or co-authorship) and compute centrality metrics.
2. Use the karate club graph: compute eigenvector centrality and interpret top nodes.
3. Try community detection with other algorithms (Louvain via `community` package) and compare.
4. Build an interactive PyVis visualization and customize node tooltips and colors.
5. (Optional) Install `keplermapper` and run a Mapper pipeline; compare with the simplified Mapper above.

Deliverable: a short notebook with visualizations, metrics, and interpretation of communities.